# Chat Max 100K — Resumable Google Colab Run

This notebook trains the existing from-scratch GPT-style chat model on Google Colab.

## Colab setup
1. Upload this notebook to Google Colab and open it.
2. Select **Runtime → Change runtime type → T4 GPU** (if available), then Save.
3. Run the notebook from the top. Approve Google Drive access when prompted.
4. Put these three files in `MyDrive/chat_max_100k/data/`:
   - `chat_train.jsonl`
   - `chat_validation.jsonl`
   - `tokenizer_chat.json`
5. Run all cells. The notebook checks that CUDA is available and stops if not.

## Checkpoint behavior
- **Resumable** — model, optimizer, scheduler, AMP scaler, RNG and progress are saved to `chat_100k_last.pt` every 100 optimizer steps and at every epoch.
- **Best model** — `chat_100k_best.pt` stores the model with the lowest validation loss seen so far. It is updated when validation improves; otherwise the previous best is retained.
- **Final export** — `chat_100k_final.pt` is an inference-ready export.
- Data and checkpoints live on Google Drive, not temporary Colab storage.

| Setting | Value |
|---|---|
| Data | 100k target per-turn training samples / up to 8k validation samples |
| Model | d512 / 8 heads / 10 layers / dff 2048 / sequence length 512 (~40M parameters) |
| Training | 12 epochs, effective batch 8, lr 3e-4, FP16 AMP |
| Target hardware | Colab T4 16GB, subject to free-tier availability |

**Important:** Colab Free sessions can end at any time. Checkpoints let you resume, but cannot prevent a disconnect.


## 0. Notebook version

In [ ]:
# ============================================================
# SECTION 0 — NOTEBOOK VERSION
# ============================================================

# Bump NOTEBOOK_VERSION whenever the config or logic changes, so any
# checkpoint can be traced back to the code that produced it.
NOTEBOOK_VERSION = "1.1"

VERSION_CHANGES = (
    "100k per-turn samples | resumable mid-epoch + epoch checkpoints | "
    "atomic Drive checkpoints every 100 steps + epoch | best-so-far model | Colab T4"
)

print("Notebook version :", NOTEBOOK_VERSION)
print("Changes          :", VERSION_CHANGES)

import torch
import tokenizers

print("torch            :", torch.__version__)
print("tokenizers       :", tokenizers.__version__)
print("CUDA available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))

## 1. Setup

Imports, reproducibility seeds, device selection, and fp16 autocast flag.

In [ ]:
# ============================================================
# SECTION 1 — SETUP
# ============================================================

import os
import math
import json
import random
import shutil
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader

from tokenizers import Tokenizer

# Reproducibility: fix all random seeds so resumed runs match an unbroken run
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device: use the GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Mixed precision: fp16 autocast + GradScaler (big speedup on T4)
USE_AMP = torch.cuda.is_available()

print("PyTorch:", torch.__version__)
print("Device:", device)

# This run is intended for Colab GPU training. Avoid silently training on CPU.
COLAB_GPU_REQUIRED = True
if COLAB_GPU_REQUIRED and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. In Colab choose Runtime > Change runtime type "
        "> T4 GPU, save, then rerun this notebook from the top."
    )

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

## 2. Storage — Google Drive (Colab) or local folders

On Colab this mounts your Drive and keeps everything under
`MyDrive/chat_max_100k/`:

```
MyDrive/chat_max_100k/
├── data/
│   ├── chat_train.jsonl
│   ├── chat_validation.jsonl
│   └── tokenizer_chat.json      (auto-copied, falls back to ./tokenizer_chat.json)
├── tokenizer_chat.json          (auto-copied here so chatbot_core.py can load it)
└── checkpoints/
    ├── chat_100k_last.pt        <- resumable state (written mid-epoch + every epoch)
    ├── chat_100k_best.pt        <- best validation loss (every epoch)
    └── chat_100k_final.pt       <- export for inference / chatbot_core.py
```

Outside Colab the notebook falls back to `./data` and `./checkpoints` relative
to the current working directory.

If you upload the three data files with Colab's Files-panel button (which saves
them to the session disk, `/content/...`, not Drive), the storage cell finds and
copies them into `MyDrive/chat_max_100k/data/` automatically — so you only have
to upload once.

### Before running the data-loading cells

In Google Drive, create `MyDrive/chat_max_100k/data/` and upload these exact files:

```text
chat_train.jsonl
chat_validation.jsonl
tokenizer_chat.json
```

The storage cell mounts Drive and expects those filenames. The dataset and tokenizer are separate required files; uploading only the notebook is not enough. The notebook creates the `checkpoints/` folder automatically.


In [ ]:
# ============================================================
# SECTION 2 — STORAGE PATHS
# ============================================================

IN_COLAB = False

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

DRIVE_ROOT = "/content/drive/MyDrive/chat_max_100k"

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        # Colab sometimes throws "credential propagation was unsuccessful" even
        # though Drive is already mounted (e.g. via the Files sidebar). Fall
        # through and keep using the existing mount when it is there.
        print("[warn] drive.mount() failed:", exc)
        print("[warn] checking for an existing /content/drive mount ...")

    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Google Drive is not mounted. Fix options:\n"
            "  1) Re-run this cell and click 'Connect to Google Drive' "
            "(pick the account that owns the files).\n"
            "  2) Click the Drive icon in the Files sidebar to mount manually.\n"
            "  3) Runtime > Restart session, then re-run this cell.\n"
            "  4) Enable third-party cookies / sign out of other Google accounts."
        )

    DATA_DIR = f"{DRIVE_ROOT}/data"
    CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
else:
    DATA_DIR = str(Path.cwd() / "data")
    CKPT_DIR = str(Path.cwd() / "checkpoints")

os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "chat_train.jsonl")
VAL_PATH = os.path.join(DATA_DIR, "chat_validation.jsonl")
TOKENIZER_PATH = os.path.join(DATA_DIR, "tokenizer_chat.json")

BEST_PATH = os.path.join(CKPT_DIR, "chat_100k_best.pt")
LAST_PATH = os.path.join(CKPT_DIR, "chat_100k_last.pt")
FINAL_PATH = os.path.join(CKPT_DIR, "chat_100k_final.pt")

# Colab's upload button puts files in the SESSION disk (/content), which is
# wiped when the runtime dies. Anything uploaded there is copied into the Drive
# data folder automatically so it survives across sessions.
os.makedirs(DATA_DIR, exist_ok=True)

SEARCH_DIRS = ["/content/data", "/content", os.getcwd(), os.path.dirname(DATA_DIR)]


def resolve_data_file(path, name):

    if os.path.exists(path):
        return path

    for folder in SEARCH_DIRS:

        candidate = os.path.join(folder, name)

        if not os.path.exists(candidate):
            continue

        if os.path.abspath(candidate) == os.path.abspath(path):
            return candidate

        try:
            shutil.copyfile(candidate, path)
            print(f"[info] found {name} in {folder}; copied it to {path}")
            return path

        except Exception as exc:
            print(f"[warn] could not copy {candidate} to {path} ({exc}); using {candidate}")
            return candidate

    return path


TRAIN_PATH = resolve_data_file(TRAIN_PATH, "chat_train.jsonl")
VAL_PATH = resolve_data_file(VAL_PATH, "chat_validation.jsonl")
TOKENIZER_PATH = resolve_data_file(TOKENIZER_PATH, "tokenizer_chat.json")

missing = [p for p in (TRAIN_PATH, VAL_PATH) if not os.path.exists(p)]

if missing:
    raise FileNotFoundError(
        "Missing data files:\n  " + "\n  ".join(missing) +
        f"\n\nUpload chat_train.jsonl and chat_validation.jsonl to:\n  {DATA_DIR}"
        "\n(or drop them into the Colab Files panel, they are copied automatically)."
    )

if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(
        f"Missing tokenizer. Upload tokenizer_chat.json to {DATA_DIR}, "
        "next to the notebook, or into the Colab Files panel."
    )

# chatbot_core.load_model() looks for the tokenizer next to the checkpoint folder,
# so keep a copy in the run's root directory.
root_tokenizer = os.path.join(os.path.dirname(CKPT_DIR), "tokenizer_chat.json")

if os.path.abspath(root_tokenizer) != os.path.abspath(TOKENIZER_PATH):
    try:
        shutil.copyfile(TOKENIZER_PATH, root_tokenizer)
    except Exception as exc:
        print(f"[warn] could not copy tokenizer to {root_tokenizer}: {exc}")

print("In Colab   :", IN_COLAB)
print("Data dir   :", DATA_DIR)
print("Ckpt dir   :", CKPT_DIR)
print("Resume file:", LAST_PATH, "(exists)" if os.path.exists(LAST_PATH) else "(none yet)")

## 3. Data & Tokenizer

The prepared conversation files live in `data/` (built by `prepare_data.py`):

- `chat_train.jsonl` — 28.3k conversations (DailyDialog + PersonaChat)
- `chat_validation.jsonl` — 2.0k conversations

Each line is `{"source": ..., "turns": [user, bot, user, bot, ...]}`.

The tokenizer is the existing 16k ByteLevel BPE plus two chat role tokens,
so `vocab_size = 16,002`.

In [ ]:
# ------------------------------------------------------------
# Conversations + tokenizer + chat special tokens
# ------------------------------------------------------------

def load_conversations(path):

    conversations = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            conversations.append(json.loads(line))

    return conversations


train_conversations = load_conversations(TRAIN_PATH)
val_conversations = load_conversations(VAL_PATH)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Special ids (0-3 are the original summarizer tokens, 16000/16001 are new)
PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")
USER_ID = tokenizer.token_to_id("<|user|>")
BOT_ID = tokenizer.token_to_id("<|bot|>")

assert None not in (PAD_ID, UNK_ID, BOS_ID, EOS_ID, USER_ID, BOT_ID)

print("Train conversations:", f"{len(train_conversations):,}")
print("Val conversations  :", f"{len(val_conversations):,}")
print("Vocabulary size    :", tokenizer.get_vocab_size())
print(
    "PAD:", PAD_ID, "| UNK:", UNK_ID, "| BOS:", BOS_ID, "| EOS:", EOS_ID,
    "| USER:", USER_ID, "| BOT:", BOT_ID
)

## 4. Per-Turn Split & Packing (the 100k part)

`chat_max.ipynb` trained one sample per *conversation* (28.3k samples). Here
every conversation is split into one sample per bot turn:

```
full conversation: [u1, b1, u2, b2, u3, b3]
  sample 1: context=[u1]          reply=b1
  sample 2: context=[u1, b1, u2]  reply=b2
  sample 3: context=[u1, b1, u2, b2, u3]  reply=b3
```

That yields ~43k DailyDialog samples + ~139k PersonaChat samples. The training
set keeps **all DailyDialog samples** and fills up to **100,000** with
(seeded-shuffled) PersonaChat samples, so everyday small-talk is well
represented while the extra PersonaChat volume provides variety.

Each sample is encoded with the same label-masking trick as before:

```
[BOS] <|user|> u1 [BOT] b1 ... <|bot|> b2 [EOS]
      ^^^^^^^^^^^^^^^^^^^^^^^^^ context   ^^^ graded
```

Samples are then **greedily packed** into fixed 512-token rows, which wastes
far less than one padded row per sample.

In [ ]:
# ------------------------------------------------------------
# Per-turn samples -> encoded rows -> datasets
# ------------------------------------------------------------

MAX_SEQ_LEN = 512
TARGET_TRAIN_SAMPLES = 100_000
MAX_VAL_SAMPLES = 8_000


def split_per_turn(conversations):

    dailydialog, personachat = [], []

    for conv in conversations:

        turns = conv["turns"]
        bucket = dailydialog if conv.get("source") == "dailydialog" else personachat

        # Bot turns sit at odd indices (turns = [user, bot, user, bot, ...])
        for i in range(1, len(turns), 2):

            reply = turns[i].strip()

            if reply:
                bucket.append({"context": turns[:i], "reply": reply})

    return dailydialog, personachat


def build_samples(conversations, target, rng):

    dailydialog, personachat = split_per_turn(conversations)

    print(f"  dailydialog samples: {len(dailydialog):,}")
    print(f"  personachat samples: {len(personachat):,}")

    # Keep all DailyDialog; fill the remaining quota with PersonaChat
    if len(dailydialog) >= target:
        chosen = rng.sample(dailydialog, target)
    else:
        rng.shuffle(personachat)
        chosen = dailydialog + personachat[: max(0, target - len(dailydialog))]

    rng.shuffle(chosen)

    print(f"  selected           : {len(chosen):,}")

    return chosen


def encode_sample(context, reply, max_len=MAX_SEQ_LEN):

    reply_ids = tokenizer.encode(reply).ids

    if not reply_ids:
        return None

    input_ids = [BOS_ID]
    labels = [-100]

    for i, text in enumerate(context):

        text_ids = tokenizer.encode(text).ids

        if not text_ids:
            continue

        marker = USER_ID if i % 2 == 0 else BOT_ID

        input_ids.extend([marker] + text_ids)
        labels.extend([-100] * (1 + len(text_ids)))

    # The bot reply (+ final EOS) is the only graded part
    input_ids.extend([BOT_ID] + reply_ids)
    labels.extend([-100] + reply_ids)

    input_ids.append(EOS_ID)
    labels.append(EOS_ID)

    # Overlong sample: drop the oldest context tokens, keep the full reply
    if len(input_ids) > max_len:

        input_ids = [BOS_ID] + input_ids[len(input_ids) - (max_len - 1):]
        labels = [-100] + labels[len(labels) - (max_len - 1):]

    return input_ids, labels


def pad_row(input_ids, labels, max_len=MAX_SEQ_LEN):

    return {
        "input_ids": list(input_ids) + [PAD_ID] * (max_len - len(input_ids)),
        "labels": list(labels) + [-100] * (max_len - len(labels))
    }


def pack_samples(samples, max_len=MAX_SEQ_LEN):

    # Greedy packing: append whole samples to the current row until the next
    # one would overflow, then start a new row. Padding only lands at row ends.
    rows = []
    cur_ids, cur_labels = [], []

    for sample in samples:

        encoded = encode_sample(sample["context"], sample["reply"], max_len)

        if encoded is None:
            continue

        input_ids, labels = encoded

        if cur_ids and len(cur_ids) + len(input_ids) > max_len:
            rows.append(pad_row(cur_ids, cur_labels, max_len))
            cur_ids, cur_labels = [], []

        cur_ids.extend(input_ids)
        cur_labels.extend(labels)

    if cur_ids:
        rows.append(pad_row(cur_ids, cur_labels, max_len))

    return rows


class ChatDataset(Dataset):

    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


def collate_fn(batch):

    input_ids = torch.tensor(
        [row["input_ids"] for row in batch], dtype=torch.long
    )
    labels = torch.tensor(
        [row["labels"] for row in batch], dtype=torch.long
    )
    attention_mask = (input_ids != PAD_ID).long()

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }


print("Building training samples (100k target):")
train_samples = build_samples(train_conversations, TARGET_TRAIN_SAMPLES, random.Random(SEED))

print("Building validation samples:")
val_samples = build_samples(val_conversations, MAX_VAL_SAMPLES, random.Random(SEED + 1))

train_rows = pack_samples(train_samples)
val_rows = pack_samples(val_samples)

graded_train = sum(sum(1 for x in row["labels"] if x != -100) for row in train_rows)
graded_val = sum(sum(1 for x in row["labels"] if x != -100) for row in val_rows)

print()
print(f"Train samples : {len(train_samples):,}")
print(f"Train rows    : {len(train_rows):,} (packed 512-token rows)")
print(f"Train graded  : {graded_train:,} supervised label tokens")
print(f"Val samples   : {len(val_samples):,}")
print(f"Val rows      : {len(val_rows):,} (packed 512-token rows)")
print(f"Val graded    : {graded_val:,} supervised label tokens")

## 5. Model (~40M parameters)

Same architecture as `chat_max.ipynb` — decoder-only, pre-LN, GELU, learned
positions, tied input/output embeddings.

| Component | Count |
|---|---|
| Token embedding 16,002 x 512 | 8.19M (tied with output head) |
| Position embedding 512 x 512 | 0.26M |
| 10 transformer blocks | 31.5M |
| **Total** | **~ 40.0M** |

In [ ]:
# ------------------------------------------------------------
# Model configuration — ~40M parameters
# ------------------------------------------------------------

VOCAB_SIZE = tokenizer.get_vocab_size()   # 16,002 (16k BPE + 2 chat tokens)

D_MODEL = 512            # hidden width
N_HEADS = 8              # attention heads (head_dim = 64)
N_LAYERS = 10            # transformer blocks
D_FF = 2048              # feed-forward inner width
DROPOUT = 0.1

assert D_MODEL % N_HEADS == 0

HEAD_DIM = D_MODEL // N_HEADS

# Full config saved inside every checkpoint (chatsbot_core.load_model reads it)
MODEL_CONFIG = {
    "vocab_size": VOCAB_SIZE,
    "max_seq_len": MAX_SEQ_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "d_ff": D_FF,
    "dropout": DROPOUT,
    "tokenizer_path": os.path.basename(TOKENIZER_PATH),
    "special_ids": {
        "pad": PAD_ID, "unk": UNK_ID, "bos": BOS_ID, "eos": EOS_ID,
        "user": USER_ID, "bot": BOT_ID
    }
}

print("Vocab:", VOCAB_SIZE, "| Seq len:", MAX_SEQ_LEN, "| Layers:", N_LAYERS)

In [ ]:
# ------------------------------------------------------------
# Embedding stack
# ------------------------------------------------------------

# Token ids -> dense vectors
class TokenEmbedding(nn.Module):

    def __init__(self, vocab_size, d_model):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, input_ids):
        return self.embedding(input_ids)


# Positions 0..T-1 -> dense vectors (learned positions, GPT-2 style)
class PositionalEmbedding(nn.Module):

    def __init__(self, max_seq_len, d_model):

        super().__init__()

        self.embedding = nn.Embedding(max_seq_len, d_model)

    def forward(self, input_ids):

        batch_size, seq_len = input_ids.shape

        positions = torch.arange(seq_len, device=input_ids.device)
        positions = positions.unsqueeze(0).expand(batch_size, -1)

        return self.embedding(positions)


# Token + position, then dropout — the standard GPT input stack
class GPTEmbedding(nn.Module):

    def __init__(self, vocab_size, d_model, max_seq_len, dropout):

        super().__init__()

        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.position_embedding = PositionalEmbedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids) + self.position_embedding(input_ids)

        return self.dropout(x)

In [ ]:
# ------------------------------------------------------------
# Multi-head causal self-attention
# ------------------------------------------------------------

# Two masks are combined before softmax:
#   causal mask  -> token i may only attend to positions <= i
#   padding mask -> padded KEYS are hidden from real tokens
# Forbidden positions are filled with -inf so softmax gives them zero weight.
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout):

        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # One fused projection produces Q, K and V in a single pass
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.output_projection = nn.Linear(d_model, d_model)

        self.attention_dropout = nn.Dropout(dropout)
        self.residual_dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):

        B, T, C = x.shape

        # (B, T, 3C) -> (B, heads, T, head_dim) for each of Q, K, V
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv.unbind(0)

        # Scaled dot-product scores: (B, heads, T, T)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Causal mask: lower-triangular True = "allowed"
        causal_mask = torch.tril(
            torch.ones(T, T, device=x.device, dtype=torch.bool)
        )
        scores = scores.masked_fill(~causal_mask, float("-inf"))

        # Padding mask: hide padded key positions
        if attention_mask is not None:

            key_mask = attention_mask[:, None, None, :].bool()

            scores = scores.masked_fill(~key_mask, float("-inf"))

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.attention_dropout(attention_weights)

        output = attention_weights @ V
        output = output.transpose(1, 2).contiguous().reshape(B, T, C)

        output = self.output_projection(output)
        output = self.residual_dropout(output)

        return output

In [ ]:
# ------------------------------------------------------------
# Feed-forward network + transformer block + stack
# ------------------------------------------------------------

# Position-wise MLP with GELU
class FeedForwardNetwork(nn.Module):

    def __init__(self, d_model, d_ff, dropout):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)


# One decoder block, PRE-LN style (GPT-2):
#   x = x + attention(LN(x));  x = x + ffn(LN(x))
class TransformerBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):

        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)

    def forward(self, x, attention_mask=None):

        x = x + self.attention(self.ln1(x), attention_mask)
        x = x + self.ffn(self.ln2(x))

        return x


# Depth of the model — a simple stack of blocks
class TransformerStack(nn.Module):

    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):

        super().__init__()

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, attention_mask=None):

        for block in self.blocks:
            x = block(x, attention_mask)

        return x

In [ ]:
# ------------------------------------------------------------
# Full model + GPT initialization
# ------------------------------------------------------------

class GPTLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_seq_len,
        d_model,
        n_heads,
        n_layers,
        d_ff,
        dropout
    ):

        super().__init__()

        self.embedding = GPTEmbedding(vocab_size, d_model, max_seq_len, dropout)
        self.transformer = TransformerStack(n_layers, d_model, n_heads, d_ff, dropout)

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: one shared matrix for input embedding and output head
        self.lm_head.weight = self.embedding.token_embedding.embedding.weight

        self.max_seq_len = max_seq_len

    def forward(self, input_ids, attention_mask=None):

        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length {T} exceeds maximum {self.max_seq_len}"
            )

        x = self.embedding(input_ids)
        x = self.transformer(x, attention_mask)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        return logits


# GPT-style init: N(0, 0.02) weights, zero biases, LN = (1, 0).
# The `initialized` set prevents double-initializing the tied weight.
def initialize_gpt_weights(model):

    initialized = set()

    for module in model.modules():

        if isinstance(module, (nn.Linear, nn.Embedding)):

            if id(module.weight) not in initialized:

                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                initialized.add(id(module.weight))

            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.LayerNorm):

            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)


model = GPTLanguageModel(
    vocab_size=MODEL_CONFIG["vocab_size"],
    max_seq_len=MODEL_CONFIG["max_seq_len"],
    d_model=MODEL_CONFIG["d_model"],
    n_heads=MODEL_CONFIG["n_heads"],
    n_layers=MODEL_CONFIG["n_layers"],
    d_ff=MODEL_CONFIG["d_ff"],
    dropout=MODEL_CONFIG["dropout"]
)

initialize_gpt_weights(model)
model.to(device)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parameters: {parameter_count:,} (~{parameter_count / 1e6:.1f}M)")
print("Device    :", next(model.parameters()).device)

## 6. Training

- **Batch 2 x accumulation 4** = effective batch 8 (VRAM-safe at seq 512)
- lr 3e-4 with 5% linear warmup + cosine decay
- weight decay only on matrix weights (never LayerNorm/embeddings)
- gradient clipping at 1.0, fp16 autocast + GradScaler
- **12 epochs** (~1.4k optimizer steps each at 100k samples)
- **Resumable**: `chat_100k_last.pt` is written every 300 optimizer steps and at
  every epoch; re-running the notebook resumes automatically.

**Smoke test first (optional):** set `SMOKE_TEST = True` to run a ~3-minute
pipeline check on 400 samples. Reset it to `False` before the real run (and
delete `chat_100k_last.pt` if you saved a smoke-test checkpoint).

In [ ]:
# ------------------------------------------------------------
# Loaders + optimization hyperparameters
# ============================================================
# SECTION 6 — TRAINING
# ------------------------------------------------------------

# Flip to True for a ~3-minute end-to-end pipeline check (tiny subsets)
SMOKE_TEST = False

BATCH_SIZE = 2                # sequences per forward/backward pass (seq 512, 40M)
GRAD_ACCUM_STEPS = 4          # effective batch = 2 x 4 = 8
EPOCHS = 12                   # passes over the training set

LEARNING_RATE = 3e-4          # good default for from-scratch small models
WEIGHT_DECAY = 0.01           # only on matrix weights, never LN/embeddings
MAX_GRAD_NORM = 1.0           # gradient clipping ceiling
PREVIEW_EVERY = 200           # optimizer steps between greedy previews
CHECKPOINT_EVERY_STEPS = 100  # optimizer steps between durable Drive checkpoints

NUM_WORKERS = 2 if IN_COLAB else 0

train_subset = ChatDataset(train_rows[:400] if SMOKE_TEST else train_rows)
val_subset = ChatDataset(val_rows[:40] if SMOKE_TEST else val_rows)


def make_train_loader(epoch):

    # A per-epoch generator makes the shuffle order depend ONLY on (SEED, epoch).
    # That is what lets a resumed run replay the exact same batch order and skip
    # the batches it already trained on.
    generator = torch.Generator()
    generator.manual_seed(SEED + 1000 + epoch)

    return DataLoader(
        train_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        generator=generator
    )


val_loader = DataLoader(
    val_subset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP
)

print("Training samples  :", len(train_subset))
print("Validation samples:", len(val_subset))
print("Batches per epoch :", math.ceil(len(train_subset) / BATCH_SIZE))
print("Optimizer steps/ep:", math.ceil(len(train_subset) / BATCH_SIZE / GRAD_ACCUM_STEPS))

In [ ]:
# ------------------------------------------------------------
# Optimizer (param groups), warmup+cosine schedule, GradScaler
# ------------------------------------------------------------

# Weight decay only on 2-D matrix weights. LayerNorm weights, all biases,
# and the embedding tables are excluded (GPT-2 convention).
decay_params = []
no_decay_params = []

for name, parameter in model.named_parameters():

    if not parameter.requires_grad:
        continue

    if parameter.ndim <= 1 or "embedding" in name:
        no_decay_params.append(parameter)

    else:
        decay_params.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0}
    ],
    lr=LEARNING_RATE,
    betas=(0.9, 0.95)
)

# LR schedule: linear warmup (5% of steps), then cosine decay to ~0
steps_per_epoch = math.ceil(len(train_subset) / BATCH_SIZE / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * 0.05))


def lr_lambda(step):

    if step < warmup_steps:
        return (step + 1) / warmup_steps

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)

    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# GradScaler handles fp16 loss scaling together with autocast
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("Steps per epoch :", steps_per_epoch)
print("Total steps     :", total_steps)
print("Warmup steps    :", warmup_steps)

### 6.1 Checkpoint utilities

Two kinds of files:

- `chat_100k_last.pt` — **everything** needed to resume: model, optimizer,
  scheduler, scaler, RNG states, epoch, in-epoch step, partial epoch losses.
- `chat_100k_best.pt` — model weights at the lowest validation loss, plus the
  full `config` block so inference can rebuild the model (and `chatbot_core.py`
  can load it directly).

In [ ]:
def atomic_torch_save(payload, path):
    """Write to a temporary file in the same Drive folder, then replace."""
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temp_path = path + ".tmp"
    torch.save(payload, temp_path)
    os.replace(temp_path, path)


def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    step_in_epoch,
    history,
    best_val,
    partial
):

    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": epoch,
        "step_in_epoch": step_in_epoch,
        "history": history,
        "best_val_loss": best_val,
        "partial": partial,
        "notebook_version": NOTEBOOK_VERSION,
        "config": MODEL_CONFIG,
        "rng": {
            "torch": torch.get_rng_state(),
            "numpy": np.random.get_state(),
            "python": random.getstate(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        }
    }

    # Do not silently fall back to /tmp: it disappears with the Colab runtime.
    atomic_torch_save(payload, path)


def save_best(path, model, epoch, val_loss, history):

    atomic_torch_save(
        {
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "history": history,
            "notebook_version": NOTEBOOK_VERSION,
            "config": MODEL_CONFIG
        },
        path
    )

### 6.2 Resume state

Set `AUTO_RESUME = True` (default) to pick up from `chat_100k_last.pt`.
The load restores optimizer moments, LR schedule position, AMP loss scale, and
all RNG streams, so a resumed epoch continues with the same shuffle order,
dropout masks, and learning rate it would have had in an unbroken run.

Resuming assumes the same data files and the same `SEED`, `TARGET_TRAIN_SAMPLES`,
`BATCH_SIZE` and `GRAD_ACCUM_STEPS` that produced the checkpoint (the batch order
depends on them). If you change any of those, delete `chat_100k_last.pt` first.
Changing `EPOCHS` is safe — it only rescales the cosine schedule.

**On Colab:** after a disconnect, reopen the notebook, run
`Runtime -> Run all` (or Sections 1-6). The storage cell mounts Drive, the
resume cell prints `RESUMED ... epoch X | step Y`, and training continues from
exactly that batch.

In [ ]:
# ------------------------------------------------------------
# Resume (or start fresh)
# ------------------------------------------------------------

AUTO_RESUME = True

start_epoch = 0
start_step = 0
best_val = float("inf")
history = {"train_loss": [], "val_loss": []}
partial_state = None

if AUTO_RESUME and os.path.exists(LAST_PATH):

    # Load on CPU: RNG byte tensors must stay CPU, and load_state_dict() will
    # cast the model/optimizer tensors to the current device for us.
    checkpoint = torch.load(LAST_PATH, map_location="cpu", weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint["epoch"])
    start_step = int(checkpoint.get("step_in_epoch", 0))
    best_val = float(checkpoint.get("best_val_loss", float("inf")))
    history = checkpoint.get("history", history)
    partial_state = checkpoint.get("partial")

    rng = checkpoint.get("rng")

    if rng is not None:
        torch.set_rng_state(rng["torch"].cpu())
        np.random.set_state(rng["numpy"])
        random.setstate(rng["python"])
        if rng.get("cuda") is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all([state.cpu() for state in rng["cuda"]])

    print("RESUMED  :", LAST_PATH)
    print("Epoch    :", start_epoch + 1, "/", EPOCHS, "| step in epoch:", start_step)
    print("Best val :", round(best_val, 4))
    print("History  :", {k: [round(x, 4) for x in v] for k, v in history.items()})

    if checkpoint.get("notebook_version") != NOTEBOOK_VERSION:
        print("[warn] checkpoint was written by notebook version",
              checkpoint.get("notebook_version"), "| current:", NOTEBOOK_VERSION)

else:
    print("STARTING FRESH — no resumable checkpoint found.")

if start_epoch >= EPOCHS:
    print("Training already completed all", EPOCHS, "epochs. Increase EPOCHS to train more.")

### 6.3 Live preview

A fixed opening user message from the validation set is answered greedily every
`PREVIEW_EVERY` optimizer steps. Watching it go from babble to chat-shaped
replies is the best early signal that training is working.

In [ ]:
# ------------------------------------------------------------
# Live preview setup
# ------------------------------------------------------------

PREVIEW_TURNS = val_conversations[10]["turns"][:1]
PREVIEW_MAX_NEW_TOKENS = 40


def build_preview_input(turns):

    input_ids = [BOS_ID]

    for i, text in enumerate(turns):
        input_ids.extend([USER_ID] if i % 2 == 0 else [BOT_ID])
        input_ids.extend(tokenizer.encode(text).ids)

    input_ids.append(BOT_ID)

    return input_ids


PREVIEW_INPUT = torch.tensor(
    [build_preview_input(PREVIEW_TURNS)],
    dtype=torch.long,
    device=device
)

# Minimal greedy generator used only for previews during training.
# Bans every token that must never be emitted as reply text.
@torch.no_grad()
def generate_preview(model, input_ids, max_new_tokens=PREVIEW_MAX_NEW_TOKENS):

    was_training = model.training

    model.eval()

    generated_ids = input_ids.clone()

    for _ in range(max_new_tokens):

        logits = model(generated_ids, torch.ones_like(generated_ids))

        next_token_logits = logits[:, -1, :].clone()

        for banned_id in (PAD_ID, UNK_ID, BOS_ID, USER_ID, BOT_ID):
            next_token_logits[:, banned_id] = float("-inf")

        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        if next_token.item() == EOS_ID:
            break

        generated_ids = torch.cat([generated_ids, next_token], dim=1)

    if was_training:
        model.train()

    reply_ids = generated_ids[0, input_ids.shape[1]:].tolist()

    return tokenizer.decode(reply_ids, skip_special_tokens=True).strip()


print("Preview prompt:", repr(PREVIEW_TURNS[0][:80]))
print("Preview prompt tokens:", PREVIEW_INPUT.shape[1])

### 6.4 train_one_epoch / evaluate_model

The training loop supports **mid-epoch resume**: `start_step` batches of the
current epoch are skipped (they were already trained on), the partial epoch loss
total is restored, and training continues on the exact same batch sequence.

In [ ]:
# ------------------------------------------------------------
# train_one_epoch
# ------------------------------------------------------------

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scheduler,
    scaler,
    device,
    epoch,
    start_step=0,
    partial=None,
    history=None,
    best_val=float("inf"),
    preview_every=PREVIEW_EVERY
):

    model.train()

    total_loss = float(partial["loss_sum"]) if partial else 0.0
    total_tokens = int(partial["tokens"]) if partial else 0
    optimizer_steps = int(partial["optimizer_steps"]) if partial else 0

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(
        train_loader,
        total=len(train_loader),
        initial=start_step,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in enumerate(progress_bar):

        # Resumed mid-epoch: replay (without model work) the batches already done
        if step < start_step:
            continue

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(input_ids, attention_mask)

            # Shift: position t predicts token t+1 (standard LM objective)
            shifted_logits = logits[:, :-1, :].contiguous()
            shifted_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shifted_logits.reshape(-1, VOCAB_SIZE),
                shifted_labels.reshape(-1),
                ignore_index=-100
            )

        scaled_loss = loss / GRAD_ACCUM_STEPS
        scaler.scale(scaled_loss).backward()

        # Per-token loss bookkeeping (comparable to validation loss)
        valid_tokens = (shifted_labels != -100).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

        is_last_batch = (step + 1) == len(train_loader)

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or is_last_batch:

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            scheduler.step()
            optimizer_steps += 1

            progress_bar.set_postfix(
                loss=f"{total_loss / max(1, total_tokens):.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}"
            )

            if optimizer_steps % preview_every == 0:

                preview = generate_preview(model, PREVIEW_INPUT)
                print(f"\n[step {optimizer_steps}] preview: {preview!r}\n")

            # Crash insurance: save the full resumable state mid-epoch.
            # Safe here because we just stepped the optimizer and cleared grads.
            if optimizer_steps % CHECKPOINT_EVERY_STEPS == 0:

                save_checkpoint(
                    LAST_PATH, model, optimizer, scheduler, scaler,
                    epoch=epoch,
                    step_in_epoch=step + 1,
                    history=history if history is not None else {"train_loss": [], "val_loss": []},
                    best_val=best_val,
                    partial={
                        "loss_sum": total_loss,
                        "tokens": total_tokens,
                        "optimizer_steps": optimizer_steps
                    }
                )

                print(f"[checkpoint] saved {os.path.basename(LAST_PATH)} at step {step + 1}/{len(train_loader)}")

    return {
        "loss": total_loss / max(1, total_tokens),
        "tokens": total_tokens,
        "optimizer_steps": optimizer_steps,
        "steps_done": len(train_loader)
    }

In [ ]:
# ------------------------------------------------------------
# evaluate_model
# ------------------------------------------------------------

# Validation loss, averaged per token so it is directly comparable to the
# training loss. Lower val loss = better next-reply prediction.
def evaluate_model(model, val_loader, device):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():

        for batch in tqdm(val_loader, desc="Validation"):

            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                logits = model(input_ids, attention_mask)

                shifted_logits = logits[:, :-1, :].contiguous()
                shifted_labels = labels[:, 1:].contiguous()

                loss = F.cross_entropy(
                    shifted_logits.reshape(-1, VOCAB_SIZE),
                    shifted_labels.reshape(-1),
                    ignore_index=-100
                )

            valid_tokens = (shifted_labels != -100).sum().item()
            total_loss += loss.item() * valid_tokens
            total_tokens += valid_tokens

    return total_loss / max(1, total_tokens)

## 6.5 Main training run

Each epoch: train -> validate -> append history -> update `chat_100k_best.pt`
if validation loss improves -> save `chat_100k_last.pt` with complete resume
state at the next epoch boundary.

The best checkpoint is intentionally **not overwritten by a worse epoch**:
it remains the lowest-validation-loss model seen so far. The latest training
state is saved separately in `chat_100k_last.pt`.


In [ ]:
# ------------------------------------------------------------
# The main run (resumable)
# ------------------------------------------------------------

for epoch in range(start_epoch, EPOCHS):

    resumed_note = " (resumed)" if (epoch == start_epoch and start_step > 0) else ""
    print(f"\nEpoch {epoch + 1}/{EPOCHS}{resumed_note}")

    train_loader = make_train_loader(epoch)

    resume_step = start_step if epoch == start_epoch else 0
    resume_partial = partial_state if epoch == start_epoch else None

    if resume_step >= len(train_loader):
        resume_step = 0
        resume_partial = None

    result = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        scaler,
        device,
        epoch=epoch,
        start_step=resume_step,
        partial=resume_partial,
        history=history,
        best_val=best_val
    )

    train_loss = result["loss"]

    val_loss = evaluate_model(model, val_loader, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")

    if val_loss < best_val:

        best_val = val_loss
        save_best(BEST_PATH, model, epoch + 1, val_loss, history)
        print(f"New best model saved -> {BEST_PATH} (val {val_loss:.4f})")

    # Epoch complete: store resumable state positioned at the start of next epoch
    save_checkpoint(
        LAST_PATH, model, optimizer, scheduler, scaler,
        epoch=epoch + 1,
        step_in_epoch=0,
        history=history,
        best_val=best_val,
        partial=None
    )

    print(f"Resumable checkpoint saved -> {LAST_PATH}")

print("\nTraining finished.")
print("Best validation loss:", round(best_val, 4))

## 7. Export & Learning Curves

The exported checkpoint stores the full config (architecture, special ids,
tokenizer path) so it can be reloaded without the notebook's globals.

In [ ]:
# ============================================================
# SECTION 7 — EXPORT FINAL CHECKPOINT
# ============================================================

source_path = BEST_PATH if os.path.exists(BEST_PATH) else LAST_PATH

if not os.path.exists(source_path):
    raise FileNotFoundError("No checkpoint found — run the training section first.")

checkpoint = torch.load(source_path, map_location="cpu", weights_only=False)

torch.save(
    {
        "model_state_dict": checkpoint["model_state_dict"],
        "epoch": checkpoint.get("epoch"),
        "val_loss": checkpoint.get("val_loss"),
        "history": checkpoint.get("history", history),
        "notebook_version": NOTEBOOK_VERSION,
        "config": MODEL_CONFIG
    },
    FINAL_PATH
)

print("Exported:", FINAL_PATH)
print("Source  :", source_path)
print("Epoch   :", checkpoint.get("epoch"))
print("Config  :", {k: v for k, v in MODEL_CONFIG.items() if k != "special_ids"})

In [ ]:
# Loss curves. Healthy run: both fall, val tracks train. If train keeps
# dropping while val flattens or rises, the model is memorizing dialogues.
plt.figure(figsize=(10, 5))

plt.plot(history["train_loss"], marker="o", label="Train Loss")
plt.plot(history["val_loss"], marker="o", label="Val Loss")

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss (per token)")
plt.title("Chat Max 100K — Resumable Run")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Generation & Chat Demo

The model is loaded from the exported checkpoint, then replies are generated
autoregressively. Role markers, BOS, PAD and UNK are banned during decoding, so
the model only produces reply text and stops at `[EOS]`.

In [ ]:
# ------------------------------------------------------------
# Rebuild the model for inference (works after a runtime restart)
# ------------------------------------------------------------

if not os.path.exists(FINAL_PATH):
    raise FileNotFoundError(
        "No exported checkpoint yet — run the training + export sections first."
    )

ckpt = torch.load(FINAL_PATH, map_location=device, weights_only=False)
cfg = ckpt["config"]

model = GPTLanguageModel(
    vocab_size=cfg["vocab_size"],
    max_seq_len=cfg["max_seq_len"],
    d_model=cfg["d_model"],
    n_heads=cfg["n_heads"],
    n_layers=cfg["n_layers"],
    d_ff=cfg["d_ff"],
    dropout=cfg["dropout"]
).to(device)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Loaded:", FINAL_PATH)
print("Epoch:", ckpt.get("epoch"), "| Notebook version:", ckpt.get("notebook_version"))
print("Val loss:", round(ckpt.get("val_loss") or float("nan"), 4))

In [ ]:
# ------------------------------------------------------------
# generate_reply — autoregressive reply generation
# ------------------------------------------------------------

@torch.no_grad()
def generate_reply(model, turns, max_new_tokens=64, temperature=0.8, top_k=50):

    model.eval()

    input_ids = [BOS_ID]

    for i, text in enumerate(turns):
        input_ids.extend([USER_ID] if i % 2 == 0 else [BOT_ID])
        input_ids.extend(tokenizer.encode(text).ids)

    input_ids.append(BOT_ID)

    generated_ids = torch.tensor(
        [input_ids[-MAX_SEQ_LEN:]],
        dtype=torch.long,
        device=device
    )

    prompt_length = generated_ids.shape[1]

    for _ in range(max_new_tokens):

        # Sliding window: never feed more tokens than the model was built for
        context_ids = generated_ids[:, -MAX_SEQ_LEN:]

        logits = model(context_ids, torch.ones_like(context_ids))

        next_token_logits = logits[:, -1, :].clone()

        for banned_id in (PAD_ID, UNK_ID, BOS_ID, USER_ID, BOT_ID):
            next_token_logits[:, banned_id] = float("-inf")

        if temperature <= 0:

            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        else:

            next_token_logits = next_token_logits / temperature

            if top_k:
                top_values, _ = torch.topk(next_token_logits, top_k, dim=-1)
                next_token_logits[
                    next_token_logits < top_values[:, [-1]]
                ] = float("-inf")

            probabilities = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)

        if next_token.item() == EOS_ID:
            break

        generated_ids = torch.cat([generated_ids, next_token], dim=1)

    reply_ids = generated_ids[0, prompt_length:].tolist()

    return tokenizer.decode(reply_ids, skip_special_tokens=True).strip()

In [ ]:
# ------------------------------------------------------------
# Scripted demo (runs headless, no input() needed)
# ------------------------------------------------------------

DEMO_PROMPTS = [
    "hi, how are you?",
    "what do you like to do for fun?",
    "do you have any pets?",
    "i love pizza. what about you?",
    "what's your favorite movie?"
]

for prompt in DEMO_PROMPTS:

    print(f"You : {prompt}")
    print(f"Bot : {generate_reply(model, [prompt], temperature=0.8, top_k=50)}")
    print()

In [ ]:
# ------------------------------------------------------------
# Interactive chat — run this cell for a live conversation
# ------------------------------------------------------------

def chat(model, max_new_tokens=64, temperature=0.8, top_k=50):

    print("Chat Max 100K — a from-scratch 40M toy bot. Type 'quit' to exit.\n")

    turns = []

    while True:

        user_text = input("You: ").strip()

        if not user_text or user_text.lower() in ("quit", "exit", "q"):
            print("Bye.")
            break

        turns.append(user_text)

        reply = generate_reply(
            model,
            turns,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k
        )

        turns.append(reply)

        print(f"Bot: {reply}\n")

        # Keep the context window bounded (drop oldest user/bot pairs)
        while (
            len(turns) > 2
            and sum(len(tokenizer.encode(t).ids) for t in turns)
            > MAX_SEQ_LEN - 80
        ):
            turns = turns[2:]


# Uncomment to start chatting:
# chat(model)

In [ ]:
# ------------------------------------------------------------
# Sanity check: continue real validation dialogues
# ------------------------------------------------------------

for idx in (0, 5, 10):

    turns = val_conversations[idx]["turns"]

    if len(turns) < 4:
        continue

    context = turns[:3]
    reference = turns[3]
    generated = generate_reply(model, context, temperature=0.8, top_k=50)

    print("=" * 70)
    print(f"VALIDATION DIALOGUE {idx}")
    print("USER      :", context[0][:140])
    print("BOT       :", context[1][:140])
    print("USER      :", context[2][:140])
    print("REFERENCE :", reference[:140])
    print("GENERATED :", generated[:140])
    print()

## 9. What To Expect

**This model will:**
- Greet and do short small talk in a conversational style
- Produce grammatical, chat-shaped replies (100k per-turn samples)
- Occasionally loop or repeat itself, especially under greedy decoding

**This model will NOT:**
- Know facts ("what is photosynthesis?" — no; the corpus has none)
- Follow instructions or answer knowledge questions
- Remember long conversations (512-token window, ~40M params)

The per-turn split means each reply is trained on *more* context variants than
the conversation-level run (`chat_max.ipynb`), so replies usually stay on-topic
a little better — but the ceiling is still "learn how people talk, not what is
true" at 40M parameters.